
# classification_upgrade.ipynb

Апгрейд делаем **поверх логики из** `sce_net_fashion_compatibility_pair_classification.ipynb`.

Что важно:
- никаких `pos_weight` и `label_smoothing` (по вашему запросу убрано полностью);
- добавляем только серию экспериментов с разной разморозкой backbone;
- конфиги упрощены и читаются явно.


In [ ]:

import copy
import random
import time
from dataclasses import dataclass, replace, asdict
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, log_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 1) Базовые сущности (переиспользованы из pair-classification ноутбука)


In [ ]:

@dataclass
class PairClsConfig:
    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1
    pair_hidden: int = 1024

    batch_size: int = 256
    lr: float = 1e-4
    weight_decay: float = 1e-4
    epochs: int = 15
    use_amp: bool = True
    freeze_backbone: bool = True

    lambda_l1: float = 1e-4
    lambda_l2: float = 1e-4

    # Для ноутбука безопаснее 0..4, чтобы не ловить multiprocessing ошибки в Jupyter
    num_workers: int = 0
    prefetch_factor: int = 2
    persistent_workers: bool = False

cfg_base = PairClsConfig()
processor = CLIPProcessor.from_pretrained(cfg_base.hf_model_name)
cfg_base


In [ ]:

from concurrent.futures import ThreadPoolExecutor

def build_image_cache(paths, target_size: int = 224, verbose: bool = True, max_workers: int = 16):
    unique_paths = sorted(set(map(str, paths)))
    def _read_one(path: str):
        img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'Cannot read image while caching: {path}')
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if target_size is not None:
            img_rgb = cv2.resize(img_rgb, (target_size, target_size), interpolation=cv2.INTER_AREA)
        return path, img_rgb

    cache = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for path, img in tqdm(ex.map(_read_one, unique_paths), total=len(unique_paths), disable=not verbose, desc='Caching images'):
            cache[path] = img
    return cache


def compute_binary_metrics(y_true: np.ndarray, y_prob: np.ndarray, thr: float = 0.5):
    y_pred = (y_prob >= thr).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'f1@0.5': f1_score(y_true, y_pred),
        'acc@0.5': accuracy_score(y_true, y_pred),
        'logloss': log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6)),
    }


In [ ]:

class SCENetEncoder(nn.Module):
    def __init__(self, clip_model_name: str, num_conditions: int = 5, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            emb_dim = proj.out_features if hasattr(proj, 'out_features') else self.clip.config.projection_dim
        else:
            emb_dim = getattr(self.clip.config, 'projection_dim', None)
            if emb_dim is None:
                raise ValueError('Cannot infer embedding dim')

        self.emb_dim = emb_dim
        self.condition_masks = nn.Parameter(torch.empty(num_conditions, emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions)
        )

    def _extract_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out = self.clip.get_image_features(pixel_values=pixel_values)
        if torch.is_tensor(out):
            return out
        if hasattr(out, 'image_embeds') and out.image_embeds is not None:
            return out.image_embeds
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            return out.pooler_output
        raise RuntimeError('Cannot parse CLIP output')

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return F.normalize(self._extract_image_features(pixel_values), p=2, dim=-1)

    def forward_pair_embeddings(self, px1: torch.Tensor, px2: torch.Tensor):
        v1 = self.encode_image(px1)
        v2 = self.encode_image(px2)
        c = torch.sigmoid(self.condition_masks)
        o1 = c.unsqueeze(0) * v1.unsqueeze(1)
        o2 = c.unsqueeze(0) * v2.unsqueeze(1)
        w = F.softmax(self.weight_branch(torch.cat([v1, v2], dim=-1)), dim=-1)
        e1 = torch.einsum('bm,bmd->bd', w, o1)
        e2 = torch.einsum('bm,bmd->bd', w, o2)
        return e1, e2


class SCEPairClassifier(nn.Module):
    def __init__(self, cfg: PairClsConfig):
        super().__init__()
        self.encoder = SCENetEncoder(cfg.hf_model_name, cfg.num_conditions, cfg.condition_hidden, cfg.dropout)
        d = self.encoder.emb_dim
        self.classifier = nn.Sequential(
            nn.Linear(4 * d, cfg.pair_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.pair_hidden, 1),
        )

    def forward(self, px1: torch.Tensor, px2: torch.Tensor):
        e1, e2 = self.encoder.forward_pair_embeddings(px1, px2)
        feat = torch.cat([e1, e2, torch.abs(e1 - e2), e1 * e2], dim=-1)
        logit = self.classifier(feat).squeeze(-1)
        return logit, e1, e2


## 2) Данные (как в базовом ноутбуке)


In [ ]:

def triplets_to_pair_df(triplets_df: pd.DataFrame) -> pd.DataFrame:
    pos_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['pos_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['pos_path'].astype(str),
        'y': 1,
    })
    neg_df = pd.DataFrame({
        'sku1': triplets_df['anchor_sku'].astype(str),
        'sku2': triplets_df['neg_sku'].astype(str),
        'sku1_path': triplets_df['anchor_path'].astype(str),
        'sku2_path': triplets_df['neg_path'].astype(str),
        'y': 0,
    })
    pair_df = pd.concat([pos_df, neg_df], axis=0, ignore_index=True)
    pair_df = pair_df.drop_duplicates(subset=['sku1','sku2','sku1_path','sku2_path','y']).reset_index(drop=True)
    pair_df = pair_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return pair_df

class PairImageDataset(Dataset):
    def __init__(self, pair_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = pair_df.reset_index(drop=True)
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.image_cache[str(row['sku1_path'])], self.image_cache[str(row['sku2_path'])], np.float32(row['y'])


def make_pair_collate(processor):
    def collate_fn(batch):
        imgs1, imgs2, ys = zip(*batch)
        px1 = processor(images=list(imgs1), return_tensors='pt')['pixel_values']
        px2 = processor(images=list(imgs2), return_tensors='pt')['pixel_values']
        ys = torch.tensor(ys, dtype=torch.float32)
        return px1, px2, ys
    return collate_fn

train_pairs = triplets_to_pair_df(train_triplets)
val_pairs = triplets_to_pair_df(val_triplets)

all_paths = list(set(pd.concat([
    train_pairs['sku1_path'], train_pairs['sku2_path'],
    val_pairs['sku1_path'], val_pairs['sku2_path'],
], axis=0).astype(str).tolist()))
IMAGE_CACHE = build_image_cache(all_paths, target_size=224, verbose=True, max_workers=16)


## 3) Training loop (максимально близко к базовому ноутбуку)


In [ ]:

def pair_classification_loss(logits, targets, e1, e2, condition_masks, lambda_l1, lambda_l2):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    l1 = condition_masks.abs().mean()
    l2 = (e1.pow(2).mean() + e2.pow(2).mean()) / 2.0
    total = bce + lambda_l1 * l1 + lambda_l2 * l2
    return total


def create_loaders(cfg: PairClsConfig):
    train_ds = PairImageDataset(train_pairs, IMAGE_CACHE)
    val_ds = PairImageDataset(val_pairs, IMAGE_CACHE)
    collate_fn = make_pair_collate(processor)

    kwargs = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
    if cfg.num_workers > 0:
        kwargs['prefetch_factor'] = cfg.prefetch_factor
        kwargs['persistent_workers'] = cfg.persistent_workers

    train_loader = DataLoader(train_ds, shuffle=True, **kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **kwargs)
    return train_loader, val_loader


def run_epoch(model, loader, cfg: PairClsConfig, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses, probs_all, y_all = [], [], []

    amp_enabled = bool(cfg.use_amp and torch.cuda.is_available())
    amp_device = 'cuda' if torch.cuda.is_available() else 'cpu'

    for px1, px2, y in tqdm(loader, leave=False):
        px1 = px1.to(device, non_blocking=True)
        px2 = px2.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast(device_type=amp_device, enabled=amp_enabled):
                logits, e1, e2 = model(px1, px2)
                loss = pair_classification_loss(logits, y, e1, e2, model.encoder.condition_masks, cfg.lambda_l1, cfg.lambda_l2)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        losses.append(loss.item())
        probs_all.append(torch.sigmoid(logits).detach().cpu().numpy())
        y_all.append(y.detach().cpu().numpy())

    y_prob = np.concatenate(probs_all)
    y_true = np.concatenate(y_all).astype(int)
    m = compute_binary_metrics(y_true, y_prob, thr=0.5)
    m['loss'] = float(np.mean(losses))
    return m


## 4) Понятные конфиги экспериментов (только разморозка backbone)


In [ ]:

@dataclass
class ExperimentConfig:
    name: str
    description: str
    lr: float
    epochs: int
    freeze_policy: str = 'freeze_all'      # freeze_all | unfreeze_all | unfreeze_last_n
    unfreeze_last_n_blocks: int = 0


def apply_freeze_policy(model: SCEPairClassifier, policy: str, unfreeze_last_n_blocks: int = 0):
    # сперва всё замораживаем
    for p in model.encoder.clip.parameters():
        p.requires_grad = False

    if policy == 'freeze_all':
        return

    if policy == 'unfreeze_all':
        for p in model.encoder.clip.parameters():
            p.requires_grad = True
        return

    if policy == 'unfreeze_last_n':
        layers = model.encoder.clip.vision_model.encoder.layers
        n = max(0, min(int(unfreeze_last_n_blocks), len(layers)))
        for layer in layers[-n:]:
            for p in layer.parameters():
                p.requires_grad = True
        if hasattr(model.encoder.clip, 'visual_projection') and model.encoder.clip.visual_projection is not None:
            for p in model.encoder.clip.visual_projection.parameters():
                p.requires_grad = True
        return

    raise ValueError(f'Unknown freeze policy: {policy}')


experiments = [
    ExperimentConfig(
        name='exp_a_freeze_baseline',
        description='База 1-в-1: backbone заморожен. Нужен как контроль.',
        lr=1e-4,
        epochs=15,
        freeze_policy='freeze_all',
    ),
    ExperimentConfig(
        name='exp_b_partial_last2',
        description='Разморозить только последние 2 блока vision encoder.',
        lr=3e-5,
        epochs=10,
        freeze_policy='unfreeze_last_n',
        unfreeze_last_n_blocks=2,
    ),
    ExperimentConfig(
        name='exp_c_partial_last4',
        description='Разморозить последние 4 блока vision encoder.',
        lr=2e-5,
        epochs=10,
        freeze_policy='unfreeze_last_n',
        unfreeze_last_n_blocks=4,
    ),
    ExperimentConfig(
        name='exp_d_unfreeze_all',
        description='Полная разморозка backbone (самый тяжёлый вариант).',
        lr=1e-5,
        epochs=8,
        freeze_policy='unfreeze_all',
    ),
]

pd.DataFrame([asdict(e) for e in experiments])


In [ ]:

# Почему раньше могли быть плохие метрики в baseline:
# 1) предыдущий notebook использовал другую логику/конфиги + pos_weight/smoothing
# 2) multiprocessing DataLoader в Jupyter мог ломать проход по эпохе
# 3) из-за этого тренировочные шаги могли идти нестабильно
#
# Здесь baseline максимально повторяет логику из base notebook и убирает лишние отличия.

results = []
for exp in experiments:
    cfg = replace(cfg_base, lr=exp.lr, epochs=exp.epochs)
    train_loader, val_loader = create_loaders(cfg)

    model = SCEPairClassifier(cfg).to(device)
    apply_freeze_policy(model, exp.freeze_policy, exp.unfreeze_last_n_blocks)

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=(cfg.use_amp and torch.cuda.is_available()))

    history = []
    best_score = -np.inf
    best_path = f"checkpoints_classification_upgrade/{exp.name}.pt"

    for epoch in range(1, cfg.epochs + 1):
        tr = run_epoch(model, train_loader, cfg, optimizer=optimizer, scaler=scaler)
        va = run_epoch(model, val_loader, cfg, optimizer=None, scaler=scaler)

        row = {'epoch': epoch, 'train_loss': tr['loss'], 'val_loss': va['loss'],
               'train_roc_auc': tr['roc_auc'], 'val_roc_auc': va['roc_auc'],
               'train_pr_auc': tr['pr_auc'], 'val_pr_auc': va['pr_auc'],
               'val_f1@0.5': va['f1@0.5'], 'val_acc@0.5': va['acc@0.5'], 'val_logloss': va['logloss']}
        history.append(row)

        if va['pr_auc'] > best_score:
            best_score = va['pr_auc']
            torch.save({'model_state': model.state_dict(), 'config': asdict(cfg), 'exp': asdict(exp)}, best_path)

        print(f"[{exp.name}] [Epoch {epoch:02d}] train pr={tr['pr_auc']:.4f} auc={tr['roc_auc']:.4f} | val pr={va['pr_auc']:.4f} auc={va['roc_auc']:.4f} f1={va['f1@0.5']:.4f}")

    results.append({'experiment': exp, 'history': pd.DataFrame(history), 'best_path': best_path, 'best_pr_auc': best_score})

leaderboard = pd.DataFrame([
    {'name': r['experiment'].name, 'description': r['experiment'].description, 'freeze_policy': r['experiment'].freeze_policy,
     'unfreeze_last_n_blocks': r['experiment'].unfreeze_last_n_blocks, 'best_path': r['best_path'], 'best_val_pr_auc': r['best_pr_auc']}
    for r in results
]).sort_values('best_val_pr_auc', ascending=False).reset_index(drop=True)
leaderboard



## Ответ на ваш вопрос про плохой baseline

Главная причина: в предыдущей версии `classification_upgrade` baseline уже **не был "чистым продолжением"** базового ноутбука.
Там накопились отличия в setup и управлении экспериментами.

В этой версии:
- код обучения возвращён максимально близко к `sce_net_fashion_compatibility_pair_classification.ipynb`;
- убраны `pos_weight` и `label_smoothing`;
- оставлены только эксперименты с разморозкой backbone;
- упрощены конфиги без `type('obj', ...)`.



## 5) Инференс на test_final_v2.csv (как в baseline-ноутбуке с классификацией)

Ниже ровно тот же паттерн инференса:
1. Загружаем `best_path`.
2. Кешируем только test-пути.
3. Восстанавливаем модель и считаем `pred_prob = sigmoid(logit)`.


In [ ]:

best_path = 'sce_net_best_classification.pt'
test = pd.read_csv('test_final_v2.csv')
print('test shape:', test.shape)
print('columns:', list(test.columns))

# Как вы и просили: кешируем только test-пути
all_paths = list(set(pd.concat([
    test['sku1_path'],
    test['sku2_path'],
], axis=0).astype(str).tolist()))

IMAGE_CACHE_TEST = build_image_cache(all_paths, target_size=224, verbose=True, max_workers=16)


In [ ]:

# Восстановление модели из чекпоинта
ckpt = torch.load(best_path, map_location=device)

# Важно: для корректного load_state_dict архитектурные параметры должны совпадать
# с теми, на которых чекпоинт был обучен (num_conditions, hidden dims, model_name, etc.)
if isinstance(ckpt, dict) and 'config' in ckpt:
    saved_cfg = ckpt['config']
    # в ckpt config мог быть сохранён как dict
    cfg_infer = PairClsConfig(**{k: v for k, v in saved_cfg.items() if k in PairClsConfig.__annotations__})
else:
    # fallback: используем текущий cfg_base
    cfg_infer = cfg_base

model = SCEPairClassifier(cfg_infer).to(device)
state = ckpt['model_state'] if isinstance(ckpt, dict) and 'model_state' in ckpt else ckpt
model.load_state_dict(state)
model.eval()
print('Loaded checkpoint:', best_path)
print('Inference cfg:', cfg_infer)


In [ ]:

@torch.no_grad()
def infer_pair_classification(
    model: SCEPairClassifier,
    pair_df: pd.DataFrame,
    processor: CLIPProcessor,
    image_cache: dict,
    batch_size: int = 128,
):
    """
    Инференс как в baseline-ноутбуке:
    - берём пары путей (sku1_path, sku2_path)
    - считаем logits
    - конвертируем в probability через sigmoid
    """
    model.eval()
    probs = []

    rows = pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist()
    for i in tqdm(range(0, len(rows), batch_size), desc='Inference'):
        chunk = rows[i:i + batch_size]
        imgs1, imgs2 = [], []

        for p1, p2 in chunk:
            if p1 not in image_cache or p2 not in image_cache:
                raise KeyError(f'Pair path not found in cache: {p1}, {p2}')
            imgs1.append(image_cache[p1])
            imgs2.append(image_cache[p2])

        px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
        px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)

        logits, _, _ = model(px1, px2)
        batch_prob = torch.sigmoid(logits)
        probs.extend(batch_prob.cpu().numpy().tolist())

    return np.array(probs, dtype=np.float32)


In [ ]:

test_pred = test.copy()
test_pred['pred_prob'] = infer_pair_classification(
    model=model,
    pair_df=test_pred,
    processor=processor,
    image_cache=IMAGE_CACHE_TEST,
    batch_size=128,
)

# Если есть target (Normal/Bad), посчитаем метрики
if 'target' in test_pred.columns:
    y_true = test_pred['target'].map({'Normal': 1, 'Bad': 0})
    valid = ~y_true.isna()
    if valid.sum() > 0:
        metrics_test = compute_binary_metrics(y_true[valid].astype(int).values, test_pred.loc[valid, 'pred_prob'].values, thr=0.5)
        print('test metrics @0.5:', metrics_test)

# Сохраняем
out_path = 'test_final_v2_predictions_upgrade.csv'
test_pred.to_csv(out_path, index=False)
print('Saved:', out_path)
test_pred.head()



### Можно ли передавать "любой" cfg в `SCEPairClassifier(cfg)` на инференсе?

Коротко: **нет, не любой**.

`cfg` на инференсе должен совпадать с архитектурой, с которой обучен чекпоинт:
- `hf_model_name`
- `num_conditions`
- `condition_hidden`
- `dropout`
- `pair_hidden`

Если эти параметры отличаются, `load_state_dict` даст mismatch по размерам/ключам.

Поэтому в этом ноутбуке сначала пытаемся взять `cfg` из `ckpt['config']`,
а только если его нет — используем fallback `cfg_base`.
